In [ ]:
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree # Change to the project directory
%env PATH=$HOME/.local/bin:$PATH
    
import json
import os
import requests
import numpy as np
import cv2
from PIL import Image
from pymongo import MongoClient
# from mmdet.apis import init_detector, inference_detector
# from mmdet.utils import register_all_modules
from data_processing.divide_photos import divide_tablet_photo
from sign_alignment.detector import ModelConfig, TabletImageDetector
from sign_alignment.data_source import LocalDataSource
from sign_alignment.visualizer import BboxVisualizer, ColorConfig

# import signs_alignment as sa
import os
import torch
from dotenv import load_dotenv

ANNOTATIONS_DIR = os.path.expanduser("~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations")
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser("~/erc-work-data/retrained_models/detr-173/epoch_1000.pth")
# # temporal change
# CHECKPOINT_FILE = os.path.expanduser("~/epoch_1000.pth")
# ANNOTATIONS_DIR = os.path.expanduser("~/filtered_annotations")
# # ---
SCORE_THRESHOLD = 0.5
OUTPUT_DIR = "alignment_results"
SAMPLE_LIMIT = 5  # number of samples to process

load_dotenv()
MONGODB_URI = os.getenv('MONGODB_URI', 'YOUR_MONGODB_URI')


In [ ]:
from sign_alignment.pipeline import (
    CropContext, PipelineConfig, DEBUG_STEPS, DEBUG_STEPS_WITH_PROTOSNAP,
    Runner, VisOptions,
)
from sign_alignment.protosnap import ProtoSnapConfig

# Set repo_root to a local clone of TAU-VAILab/ProtoSnap to enable freeze.
# Leave it as None to keep the original behavior (everything still runs).
_PS_REPO = os.path.expanduser("~/erc-src/ProtoSnap")
ps_cfg = ProtoSnapConfig(
    repo_root=_PS_REPO if os.path.isdir(_PS_REPO) else None,
    estimator="opencv",   # SD-DIFT path is a stub; OpenCV runs today
    snap_at_iter=10,      # mid-opt run-once iteration
)

model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device='auto'
)
tablet_detector = TabletImageDetector(
    model_config=model_config,
    score_threshold=SCORE_THRESHOLD,
    keep_crops=True
)

crop_context = CropContext(
    config=PipelineConfig(
        model_config=model_config,
        tablet_detector=tablet_detector,
        local_source=LocalDataSource(ANNOTATIONS_DIR),
        color_config=ColorConfig,
        output_dir=OUTPUT_DIR,
        img_idx=1,
        protosnap=ps_cfg,
    )
)

# Control which visualization outputs are produced for all steps
vis = VisOptions(info=True, display=True, save=True)

runner = Runner(
    context=crop_context,
    steps=DEBUG_STEPS_WITH_PROTOSNAP,
    vis=vis,
)

In [ ]:
import sign_alignment.pipeline as pp

runner.choose_sample(9)  # index 0 = NBC.4020, index 9 = HS.2086
# Load image, ground truth, and sign text from API in one step
runner.run_single_step(pp.step_load_data)


In [ ]:
# detect signs (full image + chosen exp_image crop)
runner.run_single_step(pp.step_detect_signs)

In [ ]:
# transform GT boxes into sub-image coordinates and visualize
runner.run_single_step(pp.step_transform_gt_to_img)


In [ ]:
# compute average detection box dimensions
runner.run_single_step(pp.step_compute_statistics)

In [ ]:
# create detection and text sub-tablets
runner.run_single_step(pp.step_create_subtablets)

In [ ]:
# DBSCAN row detection on detection sub-tablet (also reports text sub-tablet rows)
runner.run_single_step(pp.step_detect_rows)

In [ ]:
# DP row matching between detection and text sub-tablets
runner.run_single_step(pp.step_match_rows)

In [ ]:
# visualize detection rows with D# / D#→R# labels
runner.run_single_step(pp.step_visualize_detection_rows)

In [ ]:
# within-row sign matching for each matched row pair
runner.run_single_step(pp.step_match_signs_in_rows)

In [ ]:
# align text rows onto detection rows using regression baselines
# result stored in sub_tablet_aligned
runner.run_single_step(pp.step_align_text_rows)


In [ ]:
# build sign match info; draw text mapping, side-by-side composite, alignment diagnostic
runner.run_single_step(pp.step_build_sign_match_info)

In [ ]:
# position offset analysis: coarse-aligned vs detection boxes
runner.run_single_step(pp.step_offset_analysis)


In [ ]:
# ProtoSnap setup: resolve period/font, build CenterFreezeRefiner.
# Graceful no-op if repo_root or ABZ->Unicode map are missing.
runner.run_single_step(pp.step_protosnap)


In [ ]:
# create PSR optimizer and plot characteristic loss curves
runner.run_single_step(pp.step_create_psr_optimizer)

In [ ]:
# run PSR optimization (produces sub_tablet_final)
runner.run_single_step(pp.step_run_psr_optimization)

In [ ]:
# Blue = frozen centers, Orange = still being optimized.
# (Empty visualization if ProtoSnap was disabled or no signs were accepted.)
runner.run_single_step(pp.step_protosnap_freeze_vis)


In [ ]:
# Per-sign grid: prototype (left) | annotated crop (right)
# Cyan circle = original center, green/red cross = ProtoSnap center
runner.run_single_step(pp.step_protosnap_crop_vis)

In [ ]:
# optimization loss history
runner.run_single_step(pp.step_plot_loss_history)

In [ ]:
# 2x2 results comparison: coarse aligned, final optimized, det+final overlay, gt+final overlay
runner.run_single_step(pp.step_results_comparison)

In [ ]:
# analyze parameter changes between coarse-aligned and final optimized
runner.run_single_step(pp.step_param_changes)